# 08 — Bicep & Infrastructure as Code (IaC)

**Objectif :** comprendre l'**Infrastructure as Code**, découvrir **Bicep** (le langage IaC natif d'Azure), lire des exemples, puis **déployer concrètement** une petite ressource via un fichier Bicep, et **vérifier dans le portail**.

C'est la dernière brique du parcours — et c'est exactement ce qu'utilise le projet `mailroom-ia` (`projet/mailroom-ia/infra/infra-base.bicep` + `infra-apps.bicep`).

À la fin :
- vous saurez ce qu'est l'**IaC**, ses bénéfices, et les options Azure (Bicep, ARM, Terraform, Pulumi) ;
- vous lirez sans broncher un fichier `.bicep` (params, resources, modules, outputs) ;
- vous saurez déployer un Bicep avec `az deployment group create` ;
- vous aurez **déployé une vraie ressource** (un Storage Account) et l'aurez vue dans le portail Azure ;
- vous aurez **nettoyé tout le parcours** (notebooks 01-08).

> 🔌 Pré-requis : notebooks 01 → 07 complétés. Azure CLI ≥ 2.86 (Bicep est livré avec).

## 1. Qu'est-ce que l'Infrastructure as Code ?

**IaC** = décrire son infra (VM, BDD, réseau, droits…) dans des **fichiers texte versionnés**, plutôt qu'en cliquant dans un portail ou en lançant des commandes à la main.

### Avant l'IaC — le mode « clickops »

```
  👤 Admin ouvre le portail Azure
       │
       ▼
   Click click click… 30 min plus tard, une VM tourne
       │
       ▼
   3 mois après : « comment on a configuré ça déjà ? » 🤷
   En prod : « pourquoi ça marche pas comme en dev ? » 🔥
```

👎 Pas reproductible · pas versionné · documentation = mémoire collective · dérive entre environnements · risques de sécurité.

### Avec l'IaC

```
  📄 fichier infra.bicep dans Git
       │
       │ az deployment group create --template-file infra.bicep
       ▼
  ┌─────────────────────────┐
  │  Azure crée/met à jour  │
  │  exactement ce qui est  │
  │  décrit (idempotent)    │
  └─────────────────────────┘
```

👍 **Reproductible** (dev = staging = prod) · **versionné** (Git history) · **revue par PR** · **rollback facile** · **documentation vivante** · **audit** (qui a déployé quoi quand).

### IaC et DevOps

Sans IaC, pas de vrai CI/CD : on ne peut pas auto-déployer un environnement si chaque déploiement nécessite un humain qui clique. **IaC est le socle** sur lequel reposent les pipelines CI/CD modernes (cf. notebook 03 §6).

## 2. Les options IaC sur Azure

| Outil | Langage | Avantages | Limites |
|-------|---------|-----------|---------|
| **Bicep** ⭐ | DSL Microsoft (compile en ARM JSON) | Natif Azure, syntaxe propre, tooling VS Code parfait, **gratuit, pas de tfstate** à gérer | Azure only |
| **ARM Templates** | JSON | C'était l'original. Tout marche. | Verbeux, illisible |
| **Terraform** | HCL (HashiCorp Config Language) | Multi-cloud (AWS + GCP + Azure + …), énorme écosystème | State à gérer (storage account dédié), provider Azure parfois en retard sur les nouveautés |
| **Pulumi** | Python / TS / Go / C# | Vrais langages = vraie logique conditionnelle, tests unitaires | Plus complexe à démarrer |
| **Azure CLI / PowerShell** | Bash / PowerShell | Simple pour des one-shots | Pas vraiment de l'IaC : non déclaratif, pas idempotent |

### Pour ce parcours et le projet `mailroom-ia` : **Bicep**

Raisons :
- **Natif Azure** : sortie des nouveautés en même temps que les services
- **Pas de state externe** à gérer (Azure stocke l'état dans `Microsoft.Resources/deployments`)
- **Syntaxe lisible** vs ARM JSON
- **Tooling** VS Code excellent (autocomplétion, validation, navigation)
- **Gratuit**, pas de SaaS à acheter

## 3. Anatomie d'un fichier Bicep

Voici l'**exemple le plus simple possible** — créer un Storage Account :

```bicep
// hello-storage.bicep

@description('Nom du Storage Account (3-24 char, lowercase, unique globalement).')
@minLength(3)
@maxLength(24)
param storageName string

@description('Région — par défaut celle du RG.')
param location string = resourceGroup().location

resource storage 'Microsoft.Storage/storageAccounts@2024-01-01' = {
  name: storageName
  location: location
  sku: { name: 'Standard_LRS' }
  kind: 'StorageV2'
  properties: {
    minimumTlsVersion: 'TLS1_2'
    allowBlobPublicAccess: false
    supportsHttpsTrafficOnly: true
  }
}

output blobEndpoint string = storage.properties.primaryEndpoints.blob
```

**Décortiquage** :

| Élément | Rôle |
|---------|------|
| `param` | Input : ce que vous passez au déploiement (`--parameters`) |
| `@description / @minLength / @maxLength` | **Décorateurs** : validation et doc inline |
| `resource <symbol> '<type>@<apiVersion>' = { … }` | **Déclaration** d'une ressource Azure |
| `output` | Valeur renvoyée après déploiement (consommable par d'autres scripts) |
| `resourceGroup().location` | **Fonction built-in** : la région du RG cible |

### Concepts un peu plus avancés

**Conditional** — créer une ressource seulement si une condition est vraie :
```bicep
resource appi 'Microsoft.Insights/components@2020-02-02' = if (enableMonitoring) {
  name: 'appi-${name}'
  // …
}
```

**Loops** — créer N ressources :
```bicep
resource queues 'Microsoft.Storage/.../queues@…' = [for q in queueNames: {
  parent: storage
  name: q
}]
```

**Modules** — composer plusieurs fichiers Bicep :
```bicep
module storageMod 'modules/storage.bicep' = {
  name: 'storage-deploy'
  params: { storageName: 'st${suffix}' }
}
```

**Existing** — référencer une ressource créée ailleurs (ex. dans une autre passe) :
```bicep
resource existingAcr 'Microsoft.ContainerRegistry/registries@…' existing = {
  name: 'acrmailroom123'
}
```

> 💡 Tous ces patterns sont utilisés dans `projet/mailroom-ia/infra/infra-base.bicep` et `infra-apps.bicep`. Ouvrez-les après ce notebook pour voir comment ils s'enchaînent.

## 4. Le cycle de vie d'un déploiement Bicep

```
  📄 main.bicep + main.parameters.json
        │
        │  bicep build   (compile en ARM JSON, fait localement)
        ▼
  📄 main.json (ARM)
        │
        │  az deployment group validate
        ▼
  ┌──────────────────────────┐
  │  Azure ARM API           │  validation : syntaxe + droits + quotas
  └──────────────┬───────────┘
                 │ OK
                 ▼
  az deployment group what-if   ← « preview » des changements (recommandé !)
                 │
                 ▼
  az deployment group create
                 │
                 ▼
  ┌──────────────────────────┐
  │  Azure crée / met à jour │  (idempotent : ne fait que les diffs)
  │  les ressources          │
  └──────────────┬───────────┘
                 │
                 ▼
  Le déploiement est tracé dans :
  RG > Settings > Deployments → historique navigable
```

**3 commandes-clés** :

| Commande | Quoi |
|----------|------|
| `az deployment group validate` | Vérifie la syntaxe + droits, ne déploie rien |
| `az deployment group what-if` | Affiche les **changements qui seraient appliqués** (super utile avant prod) |
| `az deployment group create` | Déploie pour de vrai (idempotent) |

> 🛡️ **Bonne pratique** : toujours `what-if` avant `create` en prod. Bicep vous montre exactement ce qui va être créé, modifié, supprimé.

## 5. Hands-on — déployer un Storage Account en Bicep

On va :
1. Vérifier les pré-requis (Azure CLI + Bicep)
2. Définir nos variables (RG existant `rg-stage-<id>`)
3. Créer un fichier `hello-storage.bicep` à la volée
4. Le valider, faire un `what-if`, puis le déployer
5. Aller voir dans le portail Azure

In [ ]:
# Vérifications pré-requis
!az --version
!az bicep version

In [ ]:
# Si Bicep CLI n'est pas installé : décommentez
# !az bicep install

In [ ]:
import os, re, subprocess
from pathlib import Path

# 👇 SEULE VARIABLE À ÉDITER
RG = "rg-stage-<votre-identifiant>"

m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")

LOCATION = subprocess.check_output(
    f"az group show --name {RG} --query location -o tsv", shell=True
).decode().strip()

# Nom du storage : alphanumérique lowercase, 3-24 char, unique mondialement
clean = USER_ID.replace("-", "")
STORAGE_NAME = f"sthello{clean}"[:24]

for k, v in {"RG": RG, "USER_ID": USER_ID, "LOCATION": LOCATION,
             "STORAGE_NAME": STORAGE_NAME}.items():
    os.environ[k] = v

print(f"RG          : {RG}  (région : {LOCATION})")
print(f"Storage     : {STORAGE_NAME}")

In [ ]:
# Créer le fichier Bicep à la volée dans un dossier temporaire
DEMO = Path("./_bicep-demo")
DEMO.mkdir(exist_ok=True)

BICEP = DEMO / "hello-storage.bicep"
BICEP.write_text("""// hello-storage.bicep — minimal Bicep d'exemple

@description('Nom du Storage Account (3-24 char, lowercase, unique globalement).')
@minLength(3)
@maxLength(24)
param storageName string

@description('Région — par défaut celle du RG.')
param location string = resourceGroup().location

@description('Tags appliqués.')
param tags object = {
  project: 'mailroom-stage'
  demo: 'bicep-08'
}

resource storage 'Microsoft.Storage/storageAccounts@2024-01-01' = {
  name: storageName
  location: location
  tags: tags
  sku: { name: 'Standard_LRS' }
  kind: 'StorageV2'
  properties: {
    minimumTlsVersion: 'TLS1_2'
    allowBlobPublicAccess: false
    supportsHttpsTrafficOnly: true
  }
}

output blobEndpoint string = storage.properties.primaryEndpoints.blob
output storageId string = storage.id
""", encoding="utf-8")

print(f"✓ Bicep écrit : {BICEP.resolve()}")
print("\n--- Contenu ---\n")
print(BICEP.read_text(encoding="utf-8"))

### 5.1 Validation

On commence par **valider** le Bicep (vérifie la syntaxe + les droits sans rien déployer).

In [ ]:
!az deployment group validate \
    --resource-group $RG \
    --template-file ./_bicep-demo/hello-storage.bicep \
    --parameters storageName=$STORAGE_NAME \
    --output table

### 5.2 What-if (preview)

Affiche **exactement ce qui va être créé / modifié / supprimé**. Très utile en prod avant un `create`.

In [ ]:
!az deployment group what-if \
    --resource-group $RG \
    --template-file ./_bicep-demo/hello-storage.bicep \
    --parameters storageName=$STORAGE_NAME

### 5.3 Déploiement

On déploie pour de vrai (~30 s).

In [ ]:
!az deployment group create \
    --name hello-storage \
    --resource-group $RG \
    --template-file ./_bicep-demo/hello-storage.bicep \
    --parameters storageName=$STORAGE_NAME \
    --output table

In [ ]:
# Récupérer les outputs déclarés dans le Bicep
import json
outputs_json = subprocess.check_output(
    f"az deployment group show --resource-group {RG} --name hello-storage --query properties.outputs -o json",
    shell=True,
).decode()
for k, v in json.loads(outputs_json).items():
    print(f"  {k}: {v['value']}")

## 6. ✅ Vérification dans le portail Azure

1. Ouvrez **https://portal.azure.com**
2. Cherchez votre **Resource Group** `rg-stage-<id>` dans la barre de recherche en haut.
3. Dans la liste des ressources, vous devriez voir votre nouveau Storage Account (`sthello…`).
4. Cliquez dessus puis observez les **tags** que vous avez appliqués (`project: mailroom-stage`, `demo: bicep-08`).

**Le plus intéressant — l'historique des déploiements :**

1. Sur la page du RG : menu de gauche → **`Settings`** → **`Deployments`**.
2. Vous voyez `hello-storage` avec son statut, son timestamp, l'utilisateur qui a déployé.
3. Cliquez dessus → **`Inputs`** (les paramètres passés), **`Outputs`** (ce qui a été renvoyé), **`Template`** (le JSON ARM compilé depuis votre Bicep).

🎉 **C'est ça, l'IaC :** chaque déploiement est **tracé, auditable, ré-exécutable**.

## 7. Bonus — modifier le Bicep et relancer (idempotence)

L'IaC est **idempotent** : redéployer le même Bicep ne casse rien, ça vérifie juste que l'état correspond. Si vous changez une propriété, seul ce qui a changé est mis à jour.

Démonstration : ajoutons un tag, faisons un `what-if`, et observons que Bicep voit le diff.

In [ ]:
# Modifier le bicep — ajout d'un tag
content = BICEP.read_text(encoding="utf-8")
content = content.replace(
    "demo: 'bicep-08'\n}",
    "demo: 'bicep-08'\n  modifiedBy: 'stagiaire'\n}",
)
BICEP.write_text(content, encoding="utf-8")
print("✓ Tag ajouté dans le Bicep")

In [ ]:
# What-if : voir le diff sans déployer
!az deployment group what-if \
    --resource-group $RG \
    --template-file ./_bicep-demo/hello-storage.bicep \
    --parameters storageName=$STORAGE_NAME

Vous devriez voir une **modification** sur le storage (le nouveau tag). Pas une recréation. C'est ça l'idempotence : Bicep diffe l'état souhaité vs l'état actuel.

## 8. Aller plus loin — exemples du projet `mailroom-ia`

Le projet utilise toutes les notions vues ici. Ouvrez les fichiers et reconnaissez les patterns :

| Fichier | Ce qu'il montre |
|---------|-----------------|
| `projet/mailroom-ia/infra/infra-base.bicep` | 7+ ressources (Log Analytics, App Insights, ACR, Storage, Cosmos, Foundry, ACA Env), `param` avec validation, `output` consommés par la passe 2 |
| `projet/mailroom-ia/infra/infra-apps.bicep` | **`existing`** pour référencer les ressources de la passe 1, **`resource roleAssignment`** pour RBAC (Managed Identity → Storage / ACR / Foundry), **`identity: { type: 'SystemAssigned' }`** |
| `infra-*.parameters.json` | Format standard pour passer des params (`@parameters` au lieu de `--parameters key=value` en ligne) |

C'est aussi la pratique illustrée dans le **`setup.ipynb`** du projet : 2 passes Bicep séparées car certaines ressources (Container Apps qui pull une image) doivent attendre que d'autres (l'ACR et son contenu) existent.

## 9. 🧹 Nettoyage **final** du parcours

Le parcours formation est terminé. **Avant d'attaquer le projet `mailroom-ia`**, on libère toutes les ressources créées dans les notebooks 01-08.

⚠️ **On ne touche JAMAIS au Resource Group** — il appartient à votre équipe.

In [ ]:
# État actuel des ressources dans le RG
!az resource list --resource-group $RG --output table

### Cleanup du notebook 08 (ce notebook)

In [ ]:
# Décommentez pour exécuter
# !az storage account delete --name $STORAGE_NAME --resource-group $RG --yes

print("Décommentez ci-dessus pour supprimer le Storage Account du notebook 08.")

### Cleanup du notebook 07 (Docker / ACA)

In [ ]:
# Variables du notebook 07 (au cas où elles ne sont pas dans la session)
ACR_NAME = f"acrhello{clean}"
ACA_ENV = f"acaenv-hello-{USER_ID}"
ACA_APP = f"hello-{USER_ID}"
for k, v in {"ACR_NAME": ACR_NAME, "ACA_ENV": ACA_ENV, "ACA_APP": ACA_APP}.items():
    os.environ[k] = v

# Décommentez pour exécuter
# !az containerapp delete --name $ACA_APP --resource-group $RG --yes 2>nul
# !az containerapp env delete --name $ACA_ENV --resource-group $RG --yes 2>nul
# !az acr delete --name $ACR_NAME --resource-group $RG --yes 2>nul

print("Décommentez ci-dessus pour supprimer ACR + ACA env + Container App du notebook 07.")

### Cleanup des notebooks 01, 02, 06

In [ ]:
# Décommentez selon ce que vous avez créé

# # --- Notebook 06 : monitoring ---
# !az monitor app-insights component delete --app appi-stage-$USER_ID --resource-group $RG 2>nul
# !az monitor log-analytics workspace delete --workspace-name law-stage-$USER_ID --resource-group $RG --yes --force true 2>nul

# # --- Notebook 02 : Foundry + agents ---
# !az cognitiveservices account project delete --name aif-stage-$USER_ID --resource-group $RG --project-name my-foundry-project 2>nul
# !az cognitiveservices account delete --name aif-stage-$USER_ID --resource-group $RG 2>nul

# # --- Notebook 01 : App Service ---
# !az webapp delete --name web-stage-$USER_ID --resource-group $RG 2>nul
# !az appservice plan delete --name asp-stage-$USER_ID --resource-group $RG --yes 2>nul

print(f"⚠️  Resource Group préservé : {RG}")
print("    Décommentez ci-dessus pour libérer toutes les ressources des notebooks 01-08.")

In [ ]:
# Nettoyer le dossier de démo Bicep local
import shutil
if DEMO.exists():
    shutil.rmtree(DEMO)
    print("✓ ./_bicep-demo supprimé")

In [ ]:
# Vérification finale : ce qui reste dans le RG
!az resource list --resource-group $RG --output table

## Récap du parcours complet

Vous avez parcouru :

| # | Sujet |
|---|-------|
| 01 | Fondamentaux Azure & PaaS (AZ-900 + AI-900) + App Service + Foundry |
| 02 | Agent IA + Workflow dans Microsoft Foundry |
| 03 | Projet, DevOps, Agile & collaboration |
| 04 | Architecture web (API, frontend, backend) |
| 05 | Sécurité cloud (Entra, RBAC, Managed Identity, Key Vault) |
| 06 | Monitoring & évaluation (App Insights, OpenTelemetry, KQL) |
| 07 | Docker, containers & Azure Container Apps |
| **08** | **Bicep & Infrastructure as Code** |

Vous avez maintenant **toutes les briques** pour comprendre, déployer et faire vivre une application IA moderne sur Azure.

### 🚀 Prochain step : le projet

Direction le **projet `mailroom-ia`** dans `projet/mailroom-ia/` :

1. Lisez **`SPEC.md`** (le QUOI : périmètre, utilisateurs, jalons)
2. Lisez **`DESIGN.md`** (le COMMENT : archi, choix techno, modèle de données)
3. Ouvrez **`setup.ipynb`** et déroulez le pas-à-pas pour provisionner et déployer

Tout ce que vous avez appris (RG existant, Managed Identity, **Bicep**, Docker, ACR, ACA Apps & Jobs, KEDA, Foundry, Document Intelligence, App Insights) y est mis en pratique.

📚 Pour aller plus loin sur Bicep :
- Doc Bicep : https://learn.microsoft.com/azure/azure-resource-manager/bicep/
- **Bicep Quickstart** : https://learn.microsoft.com/azure/azure-resource-manager/bicep/quickstart-create-bicep-use-visual-studio-code
- Reference Bicep types : https://learn.microsoft.com/azure/templates/
- AVM (Azure Verified Modules) — modules réutilisables certifiés Microsoft : https://aka.ms/avm
- Bicep playground : https://aka.ms/bicepdemo